In [1]:
import os
import copy
import math
import time
import uuid
import torch
import random
import logging
import argparse
import numpy as np
import torch.nn as nn
from math import sqrt
from torch import Tensor
from pathlib import Path
from statistics import mean
import torch.optim as optim
from einops import rearrange
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from torchvision import transforms
from torch.utils.data import DataLoader
from sympy.polys.polyconfig import query
from siformer.gaussian_noise import GaussianNoise
from typing import Optional, Union, Callable, List
from torch.nn.modules.normalization import LayerNorm
from datasets.czech_slr_dataset import CzechSLRDataset
from utils import __balance_val_split, __split_of_train_sequence, __log_class_statistics, logger
from torch.nn.modules.transformer import TransformerEncoder, TransformerEncoderLayer, TransformerDecoder

In [2]:
def get_default_args():
    parser = argparse.ArgumentParser(add_help=False)

    parser.add_argument("--experiment_name", type=str, default="WLASL_spoter",
                        help="Name of the experiment after which the logs and plots will be named")
    parser.add_argument("--num_classes", type=int, default=100, help="Number of classes to be recognized by the model")
    parser.add_argument("--batch_size", type=int, default=24, help="Number of batch size")
    parser.add_argument("--num_worker", type=int, default=0, help="Number of workers")
    parser.add_argument("--num_seq_elements", type=int, default=108, # [21(hand)*2 +12(body) ]*2
                        help="Hidden dimension of the underlying Transformer model")
    parser.add_argument("--seed", type=int, default=379,
                        help="Seed with which to initialize all the random components of the training")

    # Data
    parser.add_argument("--training_set_path", type=str, default="", help="Path to the training dataset CSV file")
    parser.add_argument("--testing_set_path", type=str, default="", help="Path to the testing dataset CSV file")
    parser.add_argument("--experimental_train_split", type=float, default=None,
                        help="Determines how big a portion of the training set should be employed (intended for the "
                             "gradually enlarging training set experiment from the paper)")

    parser.add_argument("--validation_set", type=str, choices=["from-file", "split-from-train", "none"],
                        default="none",
                        help="Type of validation set construction. See README for further rederence")
    parser.add_argument("--validation_set_size", type=float, default=0.2,
                        help="Proportion of the training set to be split as validation set, if 'validation_size' is set"
                             " to 'split-from-train'")
    parser.add_argument("--validation_set_path", type=str, default="", help="Path to the validation dataset CSV file")

    # Training hyperparameters
    parser.add_argument("--epochs", type=int, default=100, help="Number of epochs to train the model for")
    parser.add_argument("--lr", type=float, default=0.0001, help="Learning rate for the model training")
    parser.add_argument("--log_freq", type=int, default=1,
                        help="Log frequency (frequency of printing all the training info)")

    # Checkpointing
    parser.add_argument("--save_checkpoints", type=bool, default=True,
                        help="Determines whether to save weights checkpoints")

    # Scheduler
    parser.add_argument("--scheduler_factor", type=int, default=0.1, help="Factor for the ReduceLROnPlateau scheduler")
    parser.add_argument("--scheduler_patience", type=int, default=5,
                        help="Patience for the ReduceLROnPlateau scheduler")

    # Gaussian noise normalization
    parser.add_argument("--gaussian_mean", type=int, default=0, help="Mean parameter for Gaussian noise layer")
    parser.add_argument("--gaussian_std", type=int, default=0.001,
                        help="Standard deviation parameter for Gaussian noise layer")

    # Visualization
    parser.add_argument("--plot_stats", type=bool, default=True,
                        help="Determines whether continuous statistics should be plotted at the end")
    parser.add_argument("--plot_lr", type=bool, default=True,
                        help="Determines whether the LR should be plotted at the end")

    # Training time
    parser.add_argument("--record_training_time", type=bool, default=False,
                        help="Determines whether continuous statistics of training time should be record")

    # Model settings
    parser.add_argument("--attn_type", type=str, default='prob', help="The attention mechanism used by the model")
    parser.add_argument("--num_enc_layers", type=int, default=3, help="Determines the number of encoder layers")
    parser.add_argument("--num_com_layers", type=int, default=1, help="Determines the number of communicating layers")
    parser.add_argument("--num_dec_layers", type=int, default=2, help="Determines the number of decoder layers")
    parser.add_argument("--FIM", type=bool, default=True, help=" ")
    parser.add_argument("--IA_encoder", type=bool, default=True, help="Determines whether input adaptive encoder will be used")
    parser.add_argument("--IA_decoder", type=bool, default=False, help="Determines whether input adaptive decoder will be used")
    parser.add_argument("--pat_enc", type=int, default=1, help="Determines the patience of encoder for earlier exist")
    parser.add_argument("--pat_dec", type=int, default=1, help="Determines the patience of decoder for earlier exist")

    return parser

In [17]:

parser = argparse.ArgumentParser("", parents=[get_default_args()], add_help=False)

# WLASL100
# parser.set_defaults(
#     experiment_name="WLASL100",
#     training_set_path="datasets/WLASL100_train_25fps.csv",
#     testing_set_path="datasets/WLASL100_val_25fps.csv",
#     validation_set="split-from-train",
#     num_classes=100,
#     IA_decoder=True,
#     num_worker=2,
#     num_com_layers=1,
#     num_enc_layers =3,
#     num_dec_layers=4,
#     patience=2
# )

# LSA64 default args
parser.set_defaults(
    experiment_name="LSA64",
    training_set_path="datasets/LSA64_60fps.csv",
    validation_set="split-from-train",
    experimental_train_split = 0.8,
    num_classes=64,
    IA_decoder=True,
    num_worker=2,
    num_com_layers=1,
    num_enc_layers =3,
    num_dec_layers=4,
    pat_enc=1,
    pat_dec=2
)

args = parser.parse_args(args=[])

g = torch.Generator()
g.manual_seed(args.seed)

transform = transforms.Compose([GaussianNoise(args.gaussian_mean, args.gaussian_std)]) 
train_set = CzechSLRDataset(args.training_set_path, transform=transform, augmentations=True)

print(f'Tổng số hàng: {len(train_set)}')

# Validation set
val_loader = None
if args.validation_set == "from-file":
    val_set = CzechSLRDataset(args.validation_set_path)
    val_loader = DataLoader(val_set, batch_size=args.batch_size, shuffle=True, generator=g,
                            num_workers=args.num_worker)
elif args.validation_set == "split-from-train":
    train_set, val_set = __balance_val_split(train_set, 0.2)
    val_set.transform = None
    val_set.augmentations = False
    val_loader = DataLoader(val_set, batch_size=args.batch_size, shuffle=True, generator=g,
                            num_workers=args.num_worker)

# Testing set
eval_loader = None
if args.testing_set_path:
    eval_set = CzechSLRDataset(args.testing_set_path)
    eval_set = DataLoader(eval_set, batch_size=args.batch_size, shuffle=True, generator=g,
                                num_workers=args.num_worker)

# Final training set refinements
if args.experimental_train_split:
    train_set = __split_of_train_sequence(train_set, args.experimental_train_split)

train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True, generator=g,
                            num_workers=args.num_worker)

print(f'Train: {len(train_loader.dataset)}')
print(f'Val:  {len(val_loader.dataset)}')
print(f'Test: {len(eval_loader.dataset) if eval_loader is not None else None}')


Tổng số hàng: 3177
Train: 2032
Val:  636
Test: None


In [ ]:
print(f'Train: {len(train_loader.dataset) if train_loader is not None else None}')
print(f'Val: {len(val_loader.dataset) if val_loader is not None else None}')
print(f'Test: {len(eval_loader.dataset) if eval_loader is not None else None}')


Train: 2032
Val: None
Test: 636


attention.py

In [13]:

class ProbMask():
    def __init__(self, B, H, L, index, scores, device="cpu"):
        _mask = torch.ones(L, scores.shape[-1], dtype=torch.bool).to(device).triu(1)
        _mask_ex = _mask[None, None, :].expand(B, H, L, scores.shape[-1])
        indicator = _mask_ex[torch.arange(B)[:, None, None],
                    torch.arange(H)[None, :, None],
                    index, :].to(device)
        self._mask = indicator.view(scores.shape).to(device)

    @property
    def mask(self):
        return self._mask


class ProbAttention(nn.Module):
    def __init__(self, mask_flag=False, factor=5, scale=None, attention_dropout=0.1, output_attention=True):
        super(ProbAttention, self).__init__()
        self.factor = factor
        self.scale = scale
        self.mask_flag = mask_flag
        self.output_attention = output_attention
        self.dropout = nn.Dropout(attention_dropout)

    def _prob_QK(self, Q, K, sample_k, n_top):  # n_top: c*ln(L_q)
        # Q [B, H, L, D]
        B, H, L_K, E = K.shape
        _, _, L_Q, _ = Q.shape

        # calculate the sampled Q_K
        K_expand = K.unsqueeze(-3).expand(B, H, L_Q, L_K, E)
        index_sample = torch.randint(L_K, (L_Q, sample_k))  # real U = U_part(factor*ln(L_k))*L_q
        K_sample = K_expand[:, :, torch.arange(L_Q).unsqueeze(1), index_sample, :]
        Q_K_sample = torch.matmul(Q.unsqueeze(-2), K_sample.transpose(-2, -1)).squeeze(-2)

        # find the Top_k query with sparisty measurement
        M = Q_K_sample.max(-1)[0] - torch.div(Q_K_sample.sum(-1), L_K)
        M_top = M.topk(n_top, sorted=False)[1]

        # use the reduced Q to calculate Q_K
        Q_reduce = Q[torch.arange(B)[:, None, None],
                   torch.arange(H)[None, :, None],
                   M_top, :]  # factor*ln(L_q)
        Q_K = torch.matmul(Q_reduce, K.transpose(-2, -1))  # factor*ln(L_q)*L_k

        return Q_K, M_top

    def _get_initial_context(self, V, L_Q):
        B, H, L_V, D = V.shape
        if not self.mask_flag:
            # V_sum = V.sum(dim=-2)
            V_sum = V.mean(dim=-2)
            contex = V_sum.unsqueeze(-2).expand(B, H, L_Q, V_sum.shape[-1]).clone()
        else:  # use mask
            assert (L_Q == L_V)  # requires that L_Q == L_V, i.e. for self-attention only
            contex = V.cumsum(dim=-2)
        return contex

    def _update_context(self, context_in, V, scores, index, L_Q, attn_mask):
        B, H, L_V, D = V.shape

        if self.mask_flag:
            attn_mask = ProbMask(B, H, L_Q, index, scores, device=V.device)
            scores.masked_fill_(attn_mask.mask, -np.inf)

        attn = torch.softmax(scores, dim=-1)  # nn.Softmax(dim=-1)(scores)

        context_in[torch.arange(B)[:, None, None],
        torch.arange(H)[None, :, None],
        index, :] = torch.matmul(attn, V).type_as(context_in)
        if self.output_attention:
            attns = (torch.ones([B, H, L_V, L_V]) / L_V).type_as(attn).to(attn.device)
            attns[torch.arange(B)[:, None, None], torch.arange(H)[None, :, None], index, :] = attn
            return (context_in, attns)
        else:
            return (context_in, None)

    def forward(self, queries, keys, values, attn_mask):
        B, L_Q, H, D = queries.shape
        _, L_K, _, _ = keys.shape

        queries = queries.transpose(2, 1)
        keys = keys.transpose(2, 1)
        values = values.transpose(2, 1)

        U_part = self.factor * np.ceil(np.log(L_K)).astype('int').item()  # c*ln(L_k)
        u = self.factor * np.ceil(np.log(L_Q)).astype('int').item()  # c*ln(L_q)

        U_part = U_part if U_part < L_K else L_K
        u = u if u < L_Q else L_Q

        scores_top, index = self._prob_QK(queries, keys, sample_k=U_part, n_top=u)

        # add scale factor
        scale = self.scale or 1. / sqrt(D)
        if scale is not None:
            scores_top = scores_top * scale
        # get the context
        context = self._get_initial_context(values, L_Q)

        # update the context with selected top_k queries
        context, attn = self._update_context(context, values, scores_top, index, L_Q, attn_mask)

        return context.transpose(2, 1).contiguous(), attn


class AttentionLayer(nn.Module):
    def __init__(self, attention, d_model, n_heads,
                 d_keys=None, d_values=None, mix=False):
        super(AttentionLayer, self).__init__()

        d_keys = d_keys or (d_model // n_heads)
        d_values = d_values or (d_model // n_heads)

        self.inner_attention = attention
        self.query_projection = nn.Linear(d_model, d_keys * n_heads)
        self.key_projection = nn.Linear(d_model, d_keys * n_heads)
        self.value_projection = nn.Linear(d_model, d_values * n_heads)
        self.out_projection = nn.Linear(d_values * n_heads, d_model)
        self.num_heads = n_heads
        self.mix = mix
        self.batch_first = None
        self._qkv_same_embed_dim = True
        self.attention_scores = None

    def forward(self, queries, keys, values, attn_mask, key_padding_mask=None, need_weights=False, is_causal=None):
        queries = queries.permute(1, 0, 2).type(dtype=torch.float32)
        keys = keys.permute(1, 0, 2).type(dtype=torch.float32)
        values = values.permute(1, 0, 2).type(dtype=torch.float32)
        # print('queries', queries.shape) = [24, 204, 42]

        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.num_heads

        queries = self.query_projection(queries).view(B, L, H, -1)
        keys = self.key_projection(keys).view(B, S, H, -1)
        values = self.value_projection(values).view(B, S, H, -1)

        out, self.attention_scores = self.inner_attention(
            queries,
            keys,
            values,
            attn_mask
        )
        # print(self.attention_scores)
        if self.mix:
            out = out.transpose(2, 1).contiguous()
        out = out.view(B, L, -1)

        out = self.out_projection(out)
        out = out.permute(1, 0, 2).type(dtype=torch.float32)

        # print(f"out from prob_spare attention: {out.shape}")
        return out,self.attention_scores


decoder.py

In [26]:
isChecked = False

class PBEEDecoder(nn.TransformerDecoder):
    __constants__ = ['norm']

    def __init__(self, decoder_layer, num_layers, norm=None, patient=1, inner_classifiers_config=None):
        super(PBEEDecoder, self).__init__(decoder_layer, num_layers, norm)
        print(f'Using custom PBEEDecoder: num_layers= {num_layers} | patient= {patient}', )
        self.patience = patient
        self.inner_classifiers = nn.ModuleList(
            [nn.Linear(inner_classifiers_config[0], inner_classifiers_config[1])
             for _ in range(num_layers)])

    def forward(self, tgt: Tensor, memory: Tensor, tgt_mask: Optional[Tensor] = None,
                memory_mask: Optional[Tensor] = None, tgt_key_padding_mask: Optional[Tensor] = None,
                memory_key_padding_mask: Optional[Tensor] = None, tgt_is_causal: Optional[bool] = None,
                memory_is_causal: bool = False, training=True) -> Tensor:

        output = tgt
        id = uuid.uuid1()

        if training or self.patience == 0:
            for i, mod in enumerate(self.layers):
                output = mod(output, memory, tgt_mask=tgt_mask,
                             memory_mask=memory_mask,
                             tgt_key_padding_mask=tgt_key_padding_mask,
                             memory_key_padding_mask=memory_key_padding_mask)
                # mod_output = output
                # if self.norm is not None:
                #     mod_output = self.norm(mod_output)
                # _ = self.inner_classifiers[i](mod_output).squeeze()
        else:
            patient_counter = 0
            patient_result = None
            calculated_layer_num = 0
            for i, mod in enumerate(self.layers):
                calculated_layer_num += 1
                output = mod(output, memory, tgt_mask=tgt_mask,
                             memory_mask=memory_mask,
                             tgt_key_padding_mask=tgt_key_padding_mask,
                             memory_key_padding_mask=memory_key_padding_mask)

                mod_output = output
                if self.norm is not None:
                    mod_output = self.norm(mod_output)

                classifier_out = self.inner_classifiers[i](mod_output).squeeze().unsqueeze(0)
                classifier_out = classifier_out.expand(1, -1, -1)

                # labels = classifier_out.detach().argmax(dim=1)
                # _, labels = torch.max(F.softmax(classifier_out, dim=1), 1)
                label = int(torch.argmax(torch.nn.functional.softmax(classifier_out, dim=2)))

                if patient_result is not None:
                    # patient_out = patient_result.detach().argmax(dim=1)
                    # _, patient_labels = torch.max(F.softmax(patient_out, dim=1), 1)
                    patient_label = int(torch.argmax(torch.nn.functional.softmax(patient_result, dim=2)))

                if (patient_result is not None) and (patient_label == label): #torch.all(label.eq(patient_label)):
                    patient_counter += 1
                else:
                    patient_counter = 0

                patient_result = classifier_out
                if patient_counter == self.patience:
                    # print("break")
                    break

        if self.norm is not None:
            output = self.norm(output)

        return output


class DecoderLayer(nn.TransformerDecoderLayer):
    """
    Edited TransformerDecoderLayer implementation omitting the redundant self-attention operation as opposed to the
    standard implementation.
    """

    def __init__(self, d_model: int, nhead: int, dim_feedforward: int = 2048, dropout: float = 0.1,
                 activation: Union[str, Callable[[Tensor], Tensor]] = F.relu, **kwargs):
        super(DecoderLayer, self).__init__(d_model, nhead, dim_feedforward, dropout, activation)
        # Change self.multihead_attn to use Pro-sparse attention
        print('Using custom DecoderLayer')

    def forward(self, tgt: torch.Tensor, memory: torch.Tensor, tgt_mask: Optional[torch.Tensor] = None,
                memory_mask: Optional[torch.Tensor] = None, tgt_key_padding_mask: Optional[torch.Tensor] = None,
                memory_key_padding_mask: Optional[torch.Tensor] = None, tgt_is_causal: Optional[bool] = False,
                memory_is_causal: Optional[bool] = False, **kwargs) -> torch.Tensor:
        global isChecked
        if not isChecked:
            isChecked = True

        tgt = tgt + self.dropout1(tgt)
        tgt = self.norm1(tgt)
        tgt2 = self.multihead_attn(tgt, memory, memory, attn_mask=memory_mask,
                                   key_padding_mask=memory_key_padding_mask)[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)

        return tgt


model.py

In [27]:

class CommunicatingEncoderLayer(nn.Module):
    """
    Một lớp Encoder tùy chỉnh thực hiện 3 giai đoạn:
    1. Self-Attention trong mỗi luồng.
    2. Cross-Attention đa hướng giữa các luồng.
    3. Feed-Forward Network.
    """

    def __init__(self, d_model_list, nhead_list, d_ff, dropout, activation, self_attn_list: List):
        super().__init__()

        # Giai đoạn 1: Self-Attention Layers
        self.self_attn_lh = self_attn_list[0]
        self.self_attn_rh = self_attn_list[1]
        self.self_attn_body = self_attn_list[2]
        self.norm1_lh = LayerNorm(d_model_list[0])
        self.norm1_rh = LayerNorm(d_model_list[1])
        self.norm1_body = LayerNorm(d_model_list[2])

        # rh truyền cho lh
        self.lh_to_rh_attn = nn.MultiheadAttention(d_model_list[0], nhead_list[0], kdim=d_model_list[1],
                                                   vdim=d_model_list[1], dropout=dropout, batch_first=False)

        # lh truyền cho rh
        self.rh_to_lh_attn = nn.MultiheadAttention(d_model_list[1], nhead_list[1], kdim=d_model_list[0],
                                                   vdim=d_model_list[0], dropout=dropout, batch_first=False)

        # Fusion layer chỉ nhận đầu ra từ một chú ý chéo
        self.lh_fusion_layer = nn.Linear(d_model_list[0], d_model_list[0])
        self.rh_fusion_layer = nn.Linear(d_model_list[1], d_model_list[1])
        self.norm2_lh = LayerNorm(d_model_list[0])
        self.norm2_rh = LayerNorm(d_model_list[1])

        # Giai đoạn 3: Feed-Forward Networks
        self.ffn_lh = nn.Sequential(nn.Linear(d_model_list[0], d_ff), activation, nn.Linear(d_ff, d_model_list[0]))
        self.ffn_rh = nn.Sequential(nn.Linear(d_model_list[1], d_ff), activation, nn.Linear(d_ff, d_model_list[1]))
        self.ffn_body = nn.Sequential(nn.Linear(d_model_list[2], d_ff), activation, nn.Linear(d_ff, d_model_list[2]))

        self.norm3_lh = LayerNorm(d_model_list[0])
        self.norm3_rh = LayerNorm(d_model_list[1])
        self.norm3_body = LayerNorm(d_model_list[2])
        self.dropout = nn.Dropout(dropout)

    def forward(self, src_list, src_mask = None, src_key_padding_mask = None):
        l_hand_x, r_hand_x, body_x = src_list[0], src_list[1], src_list[2]

        # --- 1. Self-Attention ---
        lh_self, _ = self.self_attn_lh(l_hand_x, l_hand_x, l_hand_x, attn_mask = src_mask, key_padding_mask = src_key_padding_mask)
        l_hand_x = self.norm1_lh(l_hand_x + self.dropout(lh_self))

        rh_self, _ = self.self_attn_rh(r_hand_x, r_hand_x, r_hand_x, attn_mask = src_mask, key_padding_mask = src_key_padding_mask)
        r_hand_x = self.norm1_rh(r_hand_x + self.dropout(rh_self))

        body_self, _ = self.self_attn_body(body_x, body_x, body_x, attn_mask = src_mask, key_padding_mask = src_key_padding_mask)
        body_x = self.norm1_body(body_x + self.dropout(body_self))

        lh_from_rh, _ = self.lh_to_rh_attn(l_hand_x, r_hand_x, r_hand_x)
        lh_fused = self.lh_fusion_layer(lh_from_rh)
        l_hand_x = self.norm2_lh(l_hand_x + self.dropout(lh_fused))

        rh_from_lh, _ = self.rh_to_lh_attn(query=r_hand_x,key= l_hand_x, value=l_hand_x)
        rh_fused = self.rh_fusion_layer(rh_from_lh)
        r_hand_x = self.norm2_rh(r_hand_x + self.dropout(rh_fused))

        # --- 3. Feed-Forward Network ---
        l_hand_x = self.norm3_lh(l_hand_x + self.dropout(self.ffn_lh(l_hand_x)))
        r_hand_x = self.norm3_rh(r_hand_x + self.dropout(self.ffn_rh(r_hand_x)))
        body_x = self.norm3_body(body_x + self.dropout(self.ffn_body(body_x)))

        return [l_hand_x, r_hand_x, body_x]


class CombinedEncoder(nn.Module):
    def __init__(self, d_model_list: List[int], nhead_list: List[int],
                 num_encoder_layers: int, num_comm_layers: int,
                 dim_feedforward: int, dropout: float,
                 activation: nn.Module, attn_layer_factory, patience: int = 1,
                 inner_classifiers_config: List[int] = None,
                 projections_config: List[int] = None):
        super().__init__()

        # Giai đoạn 1: Self-Attention Layers
        self.self_attn_lh = attn_layer_factory(d_model_list[0], nhead_list[0])
        self.self_attn_rh = attn_layer_factory(d_model_list[1], nhead_list[1])
        self.self_attn_body = attn_layer_factory(d_model_list[2], nhead_list[2])

        # 2) Communicating stack
        self.comm_layers = nn.ModuleList([
            CommunicatingEncoderLayer(d_model_list=d_model_list, nhead_list=nhead_list, 
                                      d_ff=dim_feedforward, dropout=dropout, activation=activation, 
                                      self_attn_list=[self.self_attn_lh, self.self_attn_rh, self.self_attn_body])
            for _ in range(num_comm_layers)
        ])

        print(f'num_comm_layers = {num_comm_layers}')

        # Norm cuối mỗi stream
        self.norm_lh = LayerNorm(d_model_list[0])
        self.norm_rh = LayerNorm(d_model_list[1])
        self.norm_body = LayerNorm(d_model_list[2])

    def forward(self, src_list: List[Tensor], src_mask: Optional[Tensor] = None,
                src_key_padding_mask: Optional[Tensor] = None, training: bool = True) -> List[Tensor]:
        l_hand_x, r_hand_x, body_x = src_list  # [L, B, D_i]

        # 2) Communicating stack
        feats = [l_hand_x, r_hand_x, body_x]
        for layer in self.comm_layers:
            feats = layer(feats, src_mask = src_mask, src_key_padding_mask = src_key_padding_mask)

        # Norm cuối
        feats[0] = self.norm_lh(feats[0])
        feats[1] = self.norm_rh(feats[1])
        feats[2] = self.norm_body(feats[2])
        return feats  # [LH, RH, Body] đã fused


class FeatureIsolatedTransformer(nn.Transformer):
    def __init__(self, d_model_list: list, nhead_list: list,
                 num_comm_layers: int,
                 num_encoder_layers: int, num_decoder_layers: int,
                 dim_feedforward: int = 2048, dropout: float = 0.1,
                 activation: nn.Module = nn.ReLU(),
                 selected_attn: str = 'prob', output_attention: str = True,
                 inner_classifiers_config: list = None, patience: int = 1, use_pyramid_encoder: bool = False,
                 distil: bool = False, projections_config: list = None,
                 IA_encoder: bool = False, IA_decoder: bool = False, 
                 device = None):  # Dùng **kwargs cho các tham số không dùng đến

        super(FeatureIsolatedTransformer, self).__init__(sum(d_model_list), nhead_list[-1], num_encoder_layers,
                                                         num_decoder_layers, dim_feedforward, dropout, activation)
        del self.encoder

        self.d_model = sum(d_model_list)
        self.d_ff =  dim_feedforward
        self.dropout = dropout
        self.num_encoder_layers = num_encoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.device = device
        self.use_pyramid_encoder = use_pyramid_encoder
        self.use_IA_encoder = IA_encoder
        self.use_IA_decoder = IA_decoder
        self.inner_classifiers_config = inner_classifiers_config
        self.projections_config = projections_config
        self.patience = patience
        self.distil = distil
        self.activation = activation
        self.selected_attn = selected_attn
        self.output_attention = output_attention

        def attn_layer_factory(d_model, n_heads):
            Attn = ProbAttention if selected_attn == 'prob' else FullAttention
            return AttentionLayer(Attn(output_attention = output_attention), d_model, n_heads, mix = False)

        # Encoder kết hợp
        self.encoder = CombinedEncoder(
            d_model_list = d_model_list,
            nhead_list = nhead_list,
            num_encoder_layers= num_encoder_layers,
            num_comm_layers = num_comm_layers,
            dim_feedforward = dim_feedforward,
            dropout = dropout,
            activation = activation,
            attn_layer_factory = attn_layer_factory,
            patience = patience,
            inner_classifiers_config = inner_classifiers_config,
            projections_config = projections_config
        )

        # --- Khởi tạo Decoder ---
        self.decoder = self.get_custom_decoder(nhead_list[-1])

    def get_custom_decoder(self, nhead):
        decoder_layer = DecoderLayer(self.d_model, nhead, self.d_ff)
        decoder_norm = LayerNorm(self.d_model)
        self.inner_classifiers_config[0] = self.d_model
        return PBEEDecoder(decoder_layer, self.num_decoder_layers, norm = decoder_norm,
                           inner_classifiers_config = self.inner_classifiers_config, patient = self.patience)

    def forward(self, src: list, tgt: Tensor, 
                src_mask: Optional[Tensor] = None, tgt_mask: Optional[Tensor] = None, memory_mask: Optional[Tensor] = None,
                src_key_padding_mask: Optional[Tensor] = None, tgt_key_padding_mask: Optional[Tensor] = None, memory_key_padding_mask: Optional[Tensor] = None,                
                src_is_causal: Optional[bool] = None, tgt_is_causal: Optional[bool] = None, memory_is_causal: bool = False,
                training:bool = True, ) -> Tensor:
        
        lh, rh, body = self.encoder(src_list=src, src_mask = src_mask,
                                    src_key_padding_mask = src_key_padding_mask,
                                    training = training)

        # Nối lại để tạo bộ nhớ hoàn chỉnh cho decoder
        full_memory = torch.cat((lh, rh, body), dim = -1) # [L, B, D_sum]

        # Gọi Decoder
        output = self.decoder(tgt, full_memory, tgt_mask = tgt_mask,
                              memory_mask = memory_mask,
                              tgt_key_padding_mask = tgt_key_padding_mask,
                              memory_key_padding_mask = memory_key_padding_mask,
                              training=training)

        return output


class SiFormer(nn.Module):
    def __init__(self, num_classes, num_hid = 108, attn_type = 'prob',
                  num_comm_layers = 1, num_enc_layers = 3, num_dec_layers = 2, patience = 1,
                 seq_len = 204, device = None, IA_encoder = True, IA_decoder = False):
        super(SiFormer, self).__init__()
        print("Feature isolated transformer")

        # self.feature_extractor = FeatureExtractor(num_hid = 108, kernel_size = 7)
        self.l_hand_embedding = nn.Parameter(self.get_encoding_table(d_model = 42))
        self.r_hand_embedding = nn.Parameter(self.get_encoding_table(d_model = 42))
        self.body_embedding   = nn.Parameter(self.get_encoding_table(d_model = 24))

        self.class_query = nn.Parameter(torch.rand(1, 1, num_hid))
        self.transformer = FeatureIsolatedTransformer(
            d_model_list = [42, 42, 24], nhead_list = [3, 3, 2, 9],
            num_comm_layers=num_comm_layers,
            num_encoder_layers = num_enc_layers, num_decoder_layers = num_dec_layers,
            selected_attn = attn_type, 
            IA_encoder = IA_encoder, IA_decoder = IA_decoder,
            inner_classifiers_config = [num_hid, num_classes],
            projections_config = [seq_len, 1],  device = device,
            patience = patience, use_pyramid_encoder = False, distil = False
        )

        self.projection = nn.Linear(num_hid, num_classes)

    def forward(self, l_hand, r_hand, body):
        training = self.training
        batch_size = l_hand.size(0)

        # (batch_size, seq_len, respected_feature_size, coordinates): (24, 204, 54, 2)
        # -> (batch_size, seq_len, feature_size):  (24, 204, 108)
        new_l_hand = l_hand.view(l_hand.size(0), l_hand.size(1), -1) # [B, L, 21, 2] -> reshape thành [B, L, 42]
        new_r_hand = r_hand.view(r_hand.size(0), r_hand.size(1), -1)
        body = body.view(body.size(0), body.size(1), -1)
        
        # (batch_size, seq_len, feature_size) : (24, 204, 108)
        # -> (seq_len, batch_size, feature_size): (204, 24, 108)
        new_l_hand = new_l_hand.permute(1, 0, 2).type(dtype = torch.float32)
        new_r_hand = new_r_hand.permute(1, 0, 2).type(dtype = torch.float32)
        new_body = body.permute(1, 0, 2).type(dtype = torch.float32)

        # feature_map = self.feature_extractor(new_inputs)
        # transformer_in = feature_map + self.pos_embedding
        l_hand_in = new_l_hand + self.l_hand_embedding  # Shape remains the same
        r_hand_in = new_r_hand + self.r_hand_embedding
        body_in = new_body + self.body_embedding

        # (seq_len, batch_size, feature_size) -> (batch_size, 1, feature_size): (24, 1, 108)
        transformer_output = self.transformer(
            [l_hand_in, r_hand_in, body_in], self.class_query.repeat(1, batch_size, 1), training = training
        ).transpose(0, 1)

        # (batch_size, 1, feature_size) -> (batch_size, num_class): (24, 100)
        out = self.projection(transformer_output).squeeze(1)
        return out

    @staticmethod
    def get_encoding_table(d_model = 108, seq_len = 204):
        torch.manual_seed(42)
        tensor_shape = (seq_len, d_model)
        frame_pos = torch.rand(tensor_shape)
        for i in range(tensor_shape[0]):
            for j in range(1, tensor_shape[1]):
                frame_pos[i, j] = frame_pos[i, j - 1]
        frame_pos = frame_pos.unsqueeze(1)  # (seq_len, 1, feature_size): (204, 1, 108)
        return frame_pos


FILE TRAIN

In [3]:
def get_default_args():
    parser = argparse.ArgumentParser(add_help=False)

    parser.add_argument("--experiment_name", type=str, default="WLASL_spoter",
                        help="Name of the experiment after which the logs and plots will be named")
    parser.add_argument("--num_classes", type=int, default=100, help="Number of classes to be recognized by the model")
    parser.add_argument("--batch_size", type=int, default=24, help="Number of batch size")
    parser.add_argument("--num_worker", type=int, default=0, help="Number of workers")
    parser.add_argument("--num_seq_elements", type=int, default=108, # [21(hand)*2 +12(body) ]*2
                        help="Hidden dimension of the underlying Transformer model")
    parser.add_argument("--seed", type=int, default=379,
                        help="Seed with which to initialize all the random components of the training")

    # Data
    parser.add_argument("--training_set_path", type=str, default="", help="Path to the training dataset CSV file")
    parser.add_argument("--testing_set_path", type=str, default="", help="Path to the testing dataset CSV file")
    parser.add_argument("--experimental_train_split", type=float, default=None,
                        help="Determines how big a portion of the training set should be employed (intended for the "
                             "gradually enlarging training set experiment from the paper)")

    parser.add_argument("--validation_set", type=str, choices=["from-file", "split-from-train", "none"],
                        default="none",
                        help="Type of validation set construction. See README for further rederence")
    parser.add_argument("--validation_set_size", type=float, default=0.2,
                        help="Proportion of the training set to be split as validation set, if 'validation_size' is set"
                             " to 'split-from-train'")
    parser.add_argument("--validation_set_path", type=str, default="", help="Path to the validation dataset CSV file")

    # Training hyperparameters
    parser.add_argument("--epochs", type=int, default=100, help="Number of epochs to train the model for")
    parser.add_argument("--lr", type=float, default=0.0001, help="Learning rate for the model training")
    parser.add_argument("--log_freq", type=int, default=1,
                        help="Log frequency (frequency of printing all the training info)")

    # Checkpointing
    parser.add_argument("--save_checkpoints", type=bool, default=True,
                        help="Determines whether to save weights checkpoints")

    # Scheduler
    parser.add_argument("--scheduler_factor", type=int, default=0.1, help="Factor for the ReduceLROnPlateau scheduler")
    parser.add_argument("--scheduler_patience", type=int, default=5,
                        help="Patience for the ReduceLROnPlateau scheduler")

    # Gaussian noise normalization
    parser.add_argument("--gaussian_mean", type=int, default=0, help="Mean parameter for Gaussian noise layer")
    parser.add_argument("--gaussian_std", type=int, default=0.001,
                        help="Standard deviation parameter for Gaussian noise layer")

    # Visualization
    parser.add_argument("--plot_stats", type=bool, default=True,
                        help="Determines whether continuous statistics should be plotted at the end")
    parser.add_argument("--plot_lr", type=bool, default=True,
                        help="Determines whether the LR should be plotted at the end")

    # Training time
    parser.add_argument("--record_training_time", type=bool, default=False,
                        help="Determines whether continuous statistics of training time should be record")

    # Model settings
    parser.add_argument("--attn_type", type=str, default='prob', help="The attention mechanism used by the model")
    parser.add_argument("--num_enc_layers", type=int, default=3, help="Determines the number of encoder layers")
    parser.add_argument("--num_com_layers", type=int, default=1, help="Determines the number of communicating layers")
    parser.add_argument("--num_dec_layers", type=int, default=2, help="Determines the number of decoder layers")
    parser.add_argument("--FIM", type=bool, default=True, help=" ")
    parser.add_argument("--IA_encoder", type=bool, default=True, help="Determines whether input adaptive encoder will be used")
    parser.add_argument("--IA_decoder", type=bool, default=False, help="Determines whether input adaptive decoder will be used")
    parser.add_argument("--pat_enc", type=int, default=1, help="Determines the patience of encoder for earlier exist")
    parser.add_argument("--pat_dec", type=int, default=1, help="Determines the patience of decoder for earlier exist")

    return parser

In [4]:
parser = argparse.ArgumentParser("", parents=[get_default_args()], add_help=False)

# WLASL100 default args
parser.set_defaults(
    experiment_name="WLASL100",
    training_set_path="datasets/WLASL100_train_25fps.csv",
    testing_set_path="datasets/WLASL100_val_25fps.csv",
    validation_set="split-from-train",
    num_classes=100,
    IA_decoder=True,
    num_worker=2,
    num_com_layers=1,
    num_enc_layers =3,
    num_dec_layers=4,
    pat_enc=1,
    pat_dec=2
)

args = parser.parse_args(args=[])

In [17]:
# Initialize all the random seeds
random.seed(args.seed)
np.random.seed(args.seed)
os.environ["PYTHONHASHSEED"] = str(args.seed)
torch.manual_seed(args.seed)
torch.cuda.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
torch.backends.cudnn.deterministic = True
g = torch.Generator()
g.manual_seed(args.seed)

# Set the output format to print into the console and save into LOG file
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(args.experiment_name + "_" + str(args.experimental_train_split).replace(".", "") + ".log")
    ]
)

In [6]:
device = torch.device("cpu")
if torch.cuda.is_available():
    print("Cuda is available: True")
    device = torch.device("cuda")

Cuda is available: True


In [18]:
# Training set
transform = transforms.Compose([GaussianNoise(args.gaussian_mean, args.gaussian_std)]) 

train_set = CzechSLRDataset(args.training_set_path, transform=transform, augmentations=True)

# Validation set
if args.validation_set == "from-file":
    val_set = CzechSLRDataset(args.validation_set_path)
    val_loader = DataLoader(val_set, batch_size=args.batch_size, shuffle=True, generator=g,
                            num_workers=args.num_worker)
elif args.validation_set == "split-from-train":
    train_set, val_set = __balance_val_split(train_set, args.validation_set_size)

    val_set.transform = None
    val_set.augmentations = False
    val_loader = DataLoader(val_set, batch_size=args.batch_size, shuffle=True, generator=g,
                            num_workers=args.num_worker)
else:
    val_loader = None

# Testing set
if args.testing_set_path:
    eval_set = CzechSLRDataset(args.testing_set_path)
    eval_loader = DataLoader(eval_set, batch_size=args.batch_size, shuffle=True, generator=g,
                                num_workers=args.num_worker)
else:
    eval_loader = None

# Final training set refinements
if args.experimental_train_split:
    train_set = __split_of_train_sequence(train_set, args.experimental_train_split)

train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True, generator=g,
                            num_workers=args.num_worker)

In [28]:
slr_model = SiFormer(num_classes=args.num_classes, 
                     num_hid=args.num_seq_elements, 
                     attn_type=args.attn_type,
                    num_comm_layers=args.num_com_layers,
                    num_enc_layers=args.num_enc_layers, 
                    num_dec_layers=args.num_dec_layers, device=device,
                    IA_encoder=args.IA_encoder, IA_decoder=args.IA_decoder,
                    patience=args.pat_dec)

total_params = sum(p.numel() for p in slr_model.parameters())
print(f"Total parameters: {total_params:,}")

# Construct the other modules | Khởi tạo hàm mất mát (loss function)
cel_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(slr_model.parameters(), lr=args.lr, betas=(0.9, 0.999), weight_decay=1e-8) # Đây là bộ tối ưu hóa (optimizer). Nó chịu trách nhiệm cập nhật trọng số của mô hình dựa trên giá trị mất mát để cải thiện hiệu suất.
# scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[60, 80], gamma=0.1)  # 40, 60, 80  Đây là bộ lập lịch tốc độ học (learning rate scheduler). Nó sẽ tự động giảm tốc độ học tại các epoch nhất định (milestones=[60, 80]) để giúp mô hình hội tụ tốt hơn.
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=0)

# Ensure that the path for checkpointing and for images both exist
Path("out-checkpoints/" + args.experiment_name + "/").mkdir(parents=True, exist_ok=True)
Path("out-img/").mkdir(parents=True, exist_ok=True)


Feature isolated transformer
num_comm_layers = 1
Using custom DecoderLayer
Using custom PBEEDecoder: num_layers= 4 | patient= 2
Total parameters: 2,718,592


In [20]:
def train_epoch(model, dataloader, criterion, optimizer, device, scheduler=None):
    pred_correct, pred_all = 0, 0
    running_loss = 0.0
    train_time_sec_list = []

    model.train()
    
    for i, data in enumerate(dataloader):
        l_hands, r_hands, bodies, labels = data

        l_hands = l_hands.to(device)
        r_hands = r_hands.to(device)
        bodies = bodies.to(device)
        labels = labels.to(device, dtype=torch.long)

        optimizer.zero_grad()
        start_time = time.time()

        outputs = model(l_hands, r_hands, bodies)

        end_time = time.time()
        train_time_sec = end_time - start_time
        train_time_sec_list.append(train_time_sec)

        loss = criterion(outputs, labels.squeeze(1))
        loss.backward()
        optimizer.step()
        running_loss += loss

        # Statistics
        _, preds = torch.max(F.softmax(outputs, dim=1), 1)
        # print(f'preds: {preds}')
        # print(f'label: {labels.view(-1)}')
        pred_correct += torch.sum(preds == labels.view(-1)).item()
        pred_all += labels.size(0)

    if scheduler:
        scheduler.step()

    avg_train_time = mean(train_time_sec_list)

    return running_loss, pred_correct, pred_all, (pred_correct / pred_all), avg_train_time


In [21]:
def evaluate(model, dataloader, device, k=5):
    pred_correct, pred_all = 0, 0
    pred_correct_topK = 0
    stats = {i: [0, 0] for i in range(100)}

    model.eval()

    with torch.no_grad():
        for i, data in enumerate(dataloader):
            l_hands, r_hands, bodies, labels = data

            l_hands = l_hands.to(device)  # [24, 204, 21, 2]
            r_hands = r_hands.to(device)  # [24, 204, 21, 2]
            bodies = bodies.to(device)  # [24, 204, 12, 2]
            labels = labels.to(device, dtype=torch.long)  # [24, 1]

            for j in range(labels.size(0)):
                l_hand = l_hands[j].unsqueeze(0)  # [1, 204, 21, 2]
                r_hand = r_hands[j].unsqueeze(0)  # [1, 204, 21, 2]
                body = bodies[j].unsqueeze(0)  # [1, 204, 12, 2]
                label = labels[j]

                output = model(l_hand, r_hand, body)
                output = output.unsqueeze(0).expand(1, -1, -1)
                pred_all += 1

                # Top 1
                if int(torch.argmax(torch.nn.functional.softmax(output, dim=2))) == int(label):
                    stats[int(label)][0] += 1
                    pred_correct += 1
                stats[int(label)][1] += 1

                # Top K
                topK= torch.topk(output, k).indices.flatten().tolist()
                if int(label[0]) in topK:
                    pred_correct_topK += 1

    return pred_correct, pred_correct_topK, pred_all


In [29]:
# MARK: TRAINING
slr_model = slr_model.to(device)
train_acc, val_acc = 0, 0
losses, train_accs, val_accs = [], [], []
lr_progress = []
top_train_acc, top_val_acc = 0, 0
checkpoint_index = 0

if args.experimental_train_split:
    logger(
        "Starting " + args.experiment_name + "_" + str(args.experimental_train_split).replace(".", "") + "...")
else:
    logger("Starting " + args.experiment_name + "...")

logger("Training using " + args.training_set_path + "...")

if args.validation_set == "from-file":
    logger("Validation using " + args.validation_set_path + "...\n\n")

total_train_time = 0
avg_train_time_sec_list = []
# for epoch in range(args.epochs):
for epoch in range(args.epochs):
    start_time = time.time()

    train_loss, _, _, train_acc, avg_train_time = train_epoch(slr_model, train_loader, cel_criterion, optimizer,
                                                                device, scheduler=scheduler)
    end_time = time.time()
    train_time = end_time - start_time

    losses.append(train_loss.item() / len(train_loader))
    train_accs.append(train_acc)

    if args.record_training_time:
        avg_train_time_sec_list.append(avg_train_time)
        total_train_time += train_time

    if val_loader:
        pred_correct, pred_correct_topK, pred_all = evaluate(slr_model, val_loader, device)
        val_acc = pred_correct/pred_all
        val_accs.append(val_acc)

    # Save checkpoints if they are best in the current subset
    if args.save_checkpoints:
        if train_acc > top_train_acc:
            top_train_acc = train_acc
            torch.save(slr_model, "out-checkpoints/" + args.experiment_name + "/checkpoint_t_" + str(
                checkpoint_index) + ".pth")

        if val_acc > top_val_acc:
            top_val_acc = val_acc
            torch.save(slr_model, "out-checkpoints/" + args.experiment_name + "/checkpoint_v_" + str(
                checkpoint_index) + ".pth")

            logger(f'Save checkpoint for [{str(epoch + 1)}] as ' + "out-checkpoints/" + args.experiment_name
                    + "/checkpoint_v_" + str(checkpoint_index) + ".pth")

    if epoch % args.log_freq == 0:
        logger(
            "[" + str(epoch + 1) + "] TRAIN  loss: " + str(train_loss.item() / len(train_loader)) + " acc: " + str(
                train_acc))
        logger(
            f"[{str(epoch + 1)}] AVG TRAIN time per sample (sec): {str(avg_train_time)} "
        )

        if val_loader:
            logger("[" + str(epoch + 1) + "] VALIDATION  acc: " + str(val_acc))

            logger("[" + str(epoch + 1) + "] VALIDATION  Top 5 acc: " + str(top_val_acc))

        logger("")

    # Reset the top accuracies on static subsets
    if epoch % 10 == 0:
        top_train_acc, top_val_acc = 0, 0
        checkpoint_index += 1

    lr_progress.append(optimizer.param_groups[0]["lr"])


Starting WLASL100...
Training using datasets/WLASL100_train_25fps.csv...
Save checkpoint for [1] as out-checkpoints/WLASL100/checkpoint_v_0.pth
[1] TRAIN  loss: 4.670136353679907 acc: 0.006640625
[1] AVG TRAIN time per sample (sec): 0.04965025465065074 
[1] VALIDATION  acc: 0.0109375
[1] VALIDATION  Top 5 acc: 0.0109375

Save checkpoint for [2] as out-checkpoints/WLASL100/checkpoint_v_1.pth
[2] TRAIN  loss: 4.331815345265041 acc: 0.061328125
[2] AVG TRAIN time per sample (sec): 0.047654526255955204 
[2] VALIDATION  acc: 0.125
[2] VALIDATION  Top 5 acc: 0.125

Save checkpoint for [3] as out-checkpoints/WLASL100/checkpoint_v_1.pth
[3] TRAIN  loss: 3.7601509450752046 acc: 0.201171875
[3] AVG TRAIN time per sample (sec): 0.047565711992923344 
[3] VALIDATION  acc: 0.28125
[3] VALIDATION  Top 5 acc: 0.28125

Save checkpoint for [4] as out-checkpoints/WLASL100/checkpoint_v_1.pth
[4] TRAIN  loss: 3.372812146338347 acc: 0.3453125
[4] AVG TRAIN time per sample (sec): 0.04872561615204143 
[4] VAL

In [30]:
top_result_top1, top_result_name_top1 = 0, ""
top_result_topk, top_result_name_topk = 0, ""
test_accs_t=[]
test_accs_v=[]

if eval_loader:
    # MARK: TESTING
    logger("\nTesting checkpointed models starting...\n")
    for i in [11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1, 0]:
        for checkpoint_id in ["t", "v"]:
            path_to_load = "out-checkpoints/" + args.experiment_name + "/checkpoint_" + checkpoint_id + "_" + str(i) + ".pth"

            if not os.path.exists(path_to_load):                    
                continue               

            tested_model = torch.load(path_to_load, weights_only=False)

            pred_correct, pred_correct_topK, pred_all = evaluate(tested_model, eval_loader, device)

            # === Top 1 ===
            eval_acc_top1 = pred_correct/pred_all

            if checkpoint_id == "v":
                test_accs_v.append(eval_acc_top1)
            else:
                test_accs_t.append(eval_acc_top1)

            if eval_acc_top1 > top_result_top1:
                top_result_top1 = eval_acc_top1
                top_result_name_top1 = args.experiment_name + "/checkpoint_" + checkpoint_id + "_" + str(i)

            # === Top K ===
            eval_acc_topk = pred_correct_topK/pred_all

            if eval_acc_topk > top_result_topk:
                top_result_topk = eval_acc_topk
                top_result_name_topk = args.experiment_name + "/checkpoint_" + checkpoint_id + "_" + str(i)

            logger(
                f"checkpoint_{checkpoint_id}_{i:<4}  ->  "
                f"Top 1: {eval_acc_top1:<10} | "
                f"Top 5: {eval_acc_topk:<10}"
            )

    path_to_load = "out-checkpoints/" + top_result_name_top1 + '.pth'
    
    logger("\nThe top result was recorded at " + str(
        top_result_top1) + " testing accuracy. The best checkpoint is " + top_result_name_top1 + ".")

logger("\nAny desired statistics have been plotted.\nThe experiment is finished.")



Testing checkpointed models starting...

checkpoint_t_10    ->  Top 1: 0.85625    | Top 5: 0.9325    
checkpoint_v_10    ->  Top 1: 0.85625    | Top 5: 0.9325    
checkpoint_t_9     ->  Top 1: 0.855      | Top 5: 0.9325    
checkpoint_v_9     ->  Top 1: 0.855      | Top 5: 0.9325    
checkpoint_t_8     ->  Top 1: 0.8575     | Top 5: 0.93      
checkpoint_v_8     ->  Top 1: 0.85375    | Top 5: 0.9325    
checkpoint_t_7     ->  Top 1: 0.85       | Top 5: 0.9325    
checkpoint_v_7     ->  Top 1: 0.8525     | Top 5: 0.9325    
checkpoint_t_6     ->  Top 1: 0.84625    | Top 5: 0.93625   
checkpoint_v_6     ->  Top 1: 0.84875    | Top 5: 0.935     
checkpoint_t_5     ->  Top 1: 0.8475     | Top 5: 0.9325    
checkpoint_v_5     ->  Top 1: 0.83875    | Top 5: 0.93125   
checkpoint_t_4     ->  Top 1: 0.84125    | Top 5: 0.9275    
checkpoint_v_4     ->  Top 1: 0.8375     | Top 5: 0.92625   
checkpoint_t_3     ->  Top 1: 0.8375     | Top 5: 0.9225    
checkpoint_v_3     ->  Top 1: 0.8375     | 

In [ ]:
s= '''import os
import argparse
import random
import logging
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from torchvision import transforms
from torch.utils.data import DataLoader
from pathlib import Path
from utils import __balance_val_split, __split_of_train_sequence, __log_class_statistics, logger
from datasets.czech_slr_dataset import CzechSLRDataset
from siformer.model import SiFormer, SpoTer
from siformer.utils import train_epoch, evaluate, evaluate_top_k
from siformer.gaussian_noise import GaussianNoise
import time
import datetime
from statistics import mean'''
s = s.split('\n')
s.sort(key=len)
for i in s:
    print(i)

In [ ]:
s